# Fase 1 — Preprocesamiento del Dataset IroSvA para Machine Learning

Este notebook prepara las tres representaciones utilizadas por LR, RF y XGBoost:

- `normal`;
- `stem`;
- `lemma`.

Se mantiene el control de calidad interno del train (duplicados exactos y conflictos de etiqueta) y la extracción de las 9 variables lingüísticas.

El test se preprocesa con las mismas reglas, pero **no se utiliza para eliminar registros del train**.

La prevención de leakage entre folds se realiza posteriormente mediante `GROUP_ID` + `StratifiedGroupKFold` en los notebooks de modelos.


## 1. Importación de librerías

In [1]:
import pandas as pd
import re
import emoji
import unicodedata
import nltk
from nltk.stem import SnowballStemmer
import spacy
from tqdm import tqdm

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## 2. Carga y auditoría del conjunto de entrenamiento por variante

Se aplica la misma política de calidad utilizada en Transformers:

- si un texto está repetido con la misma etiqueta, se conserva una sola copia;
- si el mismo texto aparece con etiquetas contradictorias, se retiran todas sus apariciones del train y se registra el conflicto;
- posteriormente se audita también la independencia train↔test.

Esto evita resolver conflictos de anotación de forma arbitraria.


In [2]:
def limpiar_train_variante(variante):
    df = pd.read_csv(
        f'../data/irosva.{variante}.training.csv'
    )
    df['VARIANTE'] = variante

    grupos = (
        df.groupby('MESSAGE', dropna=False)['IS_IRONIC']
        .agg(['size', 'nunique'])
        .reset_index()
    )

    repetidos = grupos[
        grupos['size'] > 1
    ].copy()

    conflictos = repetidos[
        repetidos['nunique'] > 1
    ].copy()

    mensajes_conflicto = set(
        conflictos['MESSAGE'].tolist()
    )

    df_conflictos = df[
        df['MESSAGE'].isin(
            mensajes_conflicto
        )
    ].copy()

    if not df_conflictos.empty:
        df_conflictos[
            'MOTIVO_EXCLUSION'
        ] = 'mismo_texto_etiquetas_contradictorias'

    # Retirar completamente conflictos.
    df = df[
        ~df['MESSAGE'].isin(
            mensajes_conflicto
        )
    ].copy()

    antes_dedup = len(df)

    # Los duplicados restantes tienen una única etiqueta.
    df = (
        df.drop_duplicates(
            subset='MESSAGE',
            keep='first',
        )
        .reset_index(drop=True)
    )

    auditoria = {
        'variante': variante,
        'n_original': (
            len(df)
            + len(df_conflictos)
            + (antes_dedup - len(df))
        ),
        'textos_repetidos': len(repetidos),
        'conflictos_texto': len(conflictos),
        'filas_retiradas_conflicto': len(df_conflictos),
        'filas_duplicadas_misma_etiqueta_retiradas': (
            antes_dedup - len(df)
        ),
        'n_tras_control_duplicados': len(df),
    }

    return df, auditoria, df_conflictos


frames_train = []
auditoria_train = []
conflictos_train = []

for variante in ['mx', 'es', 'cu']:
    df_v, info, conflictos_v = limpiar_train_variante(
        variante
    )

    frames_train.append(df_v)
    auditoria_train.append(info)

    if not conflictos_v.empty:
        conflictos_train.append(
            conflictos_v
        )

    print(
        f"{variante}: "
        f"{info['n_tras_control_duplicados']} registros finales "
        f"tras control de duplicados/conflictos"
    )


df_train = pd.concat(
    frames_train,
    ignore_index=True,
)

df_auditoria_train = pd.DataFrame(
    auditoria_train
)

df_conflictos_train = (
    pd.concat(
        conflictos_train,
        ignore_index=True,
    )
    if conflictos_train
    else pd.DataFrame()
)

print('\nAUDITORÍA TRAIN')
display(df_auditoria_train)

if not df_conflictos_train.empty:
    print('\nCONFLICTOS DE ETIQUETA RETIRADOS')
    display(
        df_conflictos_train[
            [
                'ID',
                'MESSAGE',
                'IS_IRONIC',
                'VARIANTE',
            ]
        ]
    )

print(
    f'\nTotal train tras control de calidad: {len(df_train)}'
)
print(
    f'Registros por variante: '
    f'{df_train["VARIANTE"].value_counts().to_dict()}'
)


mx: 2399 registros finales tras control de duplicados/conflictos
es: 2397 registros finales tras control de duplicados/conflictos
cu: 2400 registros finales tras control de duplicados/conflictos

AUDITORÍA TRAIN


,variante,n_original,textos_repetidos,conflictos_texto,filas_retiradas_conflicto,filas_duplicadas_misma_etiqueta_retiradas,n_tras_control_duplicados
0,mx,2400,1,0,0,1,2399
1,es,2400,2,1,2,1,2397
2,cu,2400,0,0,0,0,2400



CONFLICTOS DE ETIQUETA RETIRADOS


,ID,MESSAGE,IS_IRONIC,VARIANTE
0,3bb2e28f5c0f1a0dd7fbcbed779ff957,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,1,es
1,5353523b4ac97aff03c46cb3d11d4d6a,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,0,es



Total train tras control de calidad: 7196
Registros por variante: {'cu': 2400, 'mx': 2399, 'es': 2397}


## 3. Carga del test oficial

El test se carga únicamente para generar sus versiones `normal`, `stem` y `lemma`.

No se utiliza para modificar la composición del train.


In [3]:
def cargar_test(variante):
    df_texto = pd.read_csv(
        f'../data/irosva.{variante} - irosva.{variante}.test.csv'
    )
    df_truth = pd.read_csv(
        f'../data/irosva.{variante}.test.truth.csv'
    )

    if 'IS_IRONIC' in df_texto.columns:
        df_texto = df_texto.drop(columns=['IS_IRONIC'])

    df = df_texto.merge(
        df_truth[['ID', 'IS_IRONIC']],
        on='ID',
        how='inner',
    )

    df['VARIANTE'] = variante
    return df


df_test = pd.concat(
    [
        cargar_test('mx'),
        cargar_test('es'),
        cargar_test('cu'),
    ],
    ignore_index=True,
)

print(f'Test cargado: {len(df_test)} registros')
print(
    df_test['VARIANTE']
    .value_counts()
    .sort_index()
)


Test cargado: 1800 registros
VARIANTE
cu    600
es    600
mx    600
Name: count, dtype: int64


## 4. Extracción de variables lingüísticas

Se extraen 9 variables lingüísticas del texto original antes del preprocesamiento.
Las features son idénticas para las tres variantes de texto (normal, stem, lemma)
porque se calculan una sola vez sobre el texto sin procesar.

(González-Ibáñez et al., 2011; Joshi et al., 2015, 2016; Šandor & Bagić Babac, 2023; RAE, 2011)

In [4]:
def quitar_tildes(texto):
    """Convierte caracteres acentuados a su forma base para capturar
    variantes de escritura informal (jamás→jamas, ningún→ningun)."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

# Lista de negaciones en español
# Fuente: RAE y Asociación de Academias de la Lengua Española (2011).
# Nueva gramática básica de la lengua española.
NEGACIONES = {
    'no', 'nunca', 'jamas', 'tampoco',
    'nada', 'nadie',
    'ningun', 'ninguno', 'ninguna', 'ningunos', 'ningunas',
    'ni', 'sino', 'sin'
}

FEATURE_COLS = ['n_exc', 'n_int', 'n_may', 'n_emo', 'n_ris',
                'n_neg', 'n_elo', 'n_com', 'n_pun']

def extraer_features_linguisticos(texto):
    """Extrae 9 variables lingüísticas como conteos sobre el texto original."""
    texto_str  = str(texto)
    texto_norm = quitar_tildes(texto_str.lower())
    palabras   = re.findall(r'\b\w+\b', texto_norm)

    # 1. Exclamaciones (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_exc = texto_str.count('!') + texto_str.count('¡')
    # 2. Interrogaciones (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_int = texto_str.count('?') + texto_str.count('¿')
    # 3. Mayúsculas enfáticas: 3+ chars (Joshi et al., 2015; Šandor & Bagić Babac, 2023)
    n_may = len(re.findall(r'\b[A-ZÁÉÍÓÚÜÑ]{3,}\b', texto_str))
    # 4. Emojis (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_emo = sum(1 for c in texto_str if c in emoji.EMOJI_DATA)
    # 5. Risas (González-Ibáñez et al., 2011)
    n_ris = len(re.findall(
        r'\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xd+|lol+|jsjs)\b', texto_norm))
    # 6. Negaciones (González-Ibáñez et al., 2011; RAE, 2011)
    n_neg = sum(1 for p in palabras if p in NEGACIONES)
    # 7. Elongación: mismo carácter 3+ veces (Šandor & Bagić Babac, 2023; Joshi et al., 2016)
    n_elo = len(re.findall(r'(.)\1{2,}', texto_norm))
    # 8. Comillas (González-Ibáñez et al., 2011; Šandor & Bagić Babac, 2023)
    n_com = len(re.findall(r'["\'\u2018\u2019\u201c\u201d«»]', texto_str))
    # 9. Puntos suspensivos (Šandor & Bagić Babac, 2023; Joshi et al., 2016)
    n_pun = texto_str.count('...') + texto_str.count('\u2026')

    return [n_exc, n_int, n_may, n_emo, n_ris, n_neg, n_elo, n_com, n_pun]

In [5]:
for df_name, df in [('Train', df_train), ('Test', df_test)]:
    feats = df['MESSAGE'].apply(extraer_features_linguisticos).tolist()
    for i, col in enumerate(FEATURE_COLS):
        df[col] = [f[i] for f in feats]

print(f'Features extraídos: {FEATURE_COLS}')
print('\n=== Medias features — Train ===')
print(df_train[FEATURE_COLS].mean().round(3))
print('\n=== Medias features — Test ===')
print(df_test[FEATURE_COLS].mean().round(3))

Features extraídos: ['n_exc', 'n_int', 'n_may', 'n_emo', 'n_ris', 'n_neg', 'n_elo', 'n_com', 'n_pun']

=== Medias features — Train ===
n_exc    0.401
n_int    0.436
n_may    0.657
n_emo    0.222
n_ris    0.006
n_neg    0.686
n_elo    0.292
n_com    0.190
n_pun    0.121
dtype: float64

=== Medias features — Test ===
n_exc    0.327
n_int    0.488
n_may    0.611
n_emo    0.178
n_ris    0.003
n_neg    0.699
n_elo    0.317
n_com    0.186
n_pun    0.141
dtype: float64


## 5. Preprocesamiento base
Genera MESSAGE_CLEAN: texto limpio sin puntuación, en minúsculas,
con emojis convertidos a texto y caracteres repetidos normalizados.
Este texto es la base para las tres variantes de vectorización.

In [6]:
def preprocesar(texto):
    """Limpieza de texto: URLs, mentions, emojis a texto,
    normalización de elongación, eliminación de puntuación, minúsculas."""
    texto = re.sub(r'http\S+|www\S+', '[URL]', texto)
    texto = re.sub(r'@\w+', '[USER]', texto)
    texto = emoji.demojize(texto, language='es')
    texto = re.sub(r'(.)\1{2,}', r'\1\1', texto)
    texto = re.sub(r'[^\w\s\[\]áéíóúüñÁÉÍÓÚÜÑ]', ' ', texto)
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df_train['MESSAGE_CLEAN'] = df_train['MESSAGE'].apply(preprocesar)
df_test['MESSAGE_CLEAN']  = df_test['MESSAGE'].apply(preprocesar)
print(f'Preprocesamiento base aplicado — Train: {df_train.shape} | Test: {df_test.shape}')

for variante in ['mx', 'es', 'cu']:
    ejemplo = df_train[df_train['VARIANTE'] == variante].iloc[0]
    print(f'\n[{variante.upper()}]')
    print(f'ORIGINAL: {ejemplo["MESSAGE"]}')
    print(f'LIMPIO:   {ejemplo["MESSAGE_CLEAN"]}')

Preprocesamiento base aplicado — Train: (7196, 15) | Test: (1800, 15)

[MX]
ORIGINAL: Rica económicamente, pero muy pobre en objetividad.
LIMPIO:   rica económicamente pero muy pobre en objetividad

[ES]
ORIGINAL: @ArmandoRuido007 @ANTI_MERMA50 @JoanTarda En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣
LIMPIO:   [user] [user] [user] en vez de joan tarda van a llamarle no han tarda en callarle la boca cara_revolviéndose_de_la_risa cara_revolviéndose_de_la_risa

[CU]
ORIGINAL: magnifico
LIMPIO:   magnifico


## 6. Stemming
Aplica SnowballStemmer sobre MESSAGE_CLEAN para obtener MESSAGE_STEM.

In [7]:
stemmer = SnowballStemmer('spanish')

def aplicar_stemming(texto):
    return ' '.join(stemmer.stem(p) for p in str(texto).split())

print('Aplicando stemming...')
df_train['MESSAGE_STEM'] = df_train['MESSAGE_CLEAN'].apply(aplicar_stemming)
df_test['MESSAGE_STEM']  = df_test['MESSAGE_CLEAN'].apply(aplicar_stemming)

print('Ejemplos stem (clean → stem):')
for i in range(3):
    clean = df_train['MESSAGE_CLEAN'].iloc[i]
    stem  = df_train['MESSAGE_STEM'].iloc[i]
    print(f'  CLEAN: {clean[:70]}')
    print(f'  STEM:  {stem[:70]}')
    print()

Aplicando stemming...
Ejemplos stem (clean → stem):
  CLEAN: rica económicamente pero muy pobre en objetividad
  STEM:  ric econom per muy pobr en objet

  CLEAN: en algo tiene razón mafias hay en todo hasta en su 4t
  STEM:  en algo tien razon mafi hay en tod hast en su 4t

  CLEAN: de cuándo acá tan preocupados por la ciencia y la investigación
  STEM:  de cuand aca tan preocup por la cienci y la investig



## 7. Lematización
Aplica lematización con spaCy (es_core_news_sm) sobre MESSAGE_CLEAN
para obtener MESSAGE_LEMMA.

Si no tienes el modelo instalado ejecuta:
`!python -m spacy download es_core_news_sm`

In [8]:
print('Cargando modelo spaCy...')
nlp = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
print('Modelo cargado.')

def lematizar_corpus(textos, batch_size=256):
    lemas = []
    for doc in tqdm(nlp.pipe(textos, batch_size=batch_size),
                    total=len(textos), desc='Lematizando'):
        lemas.append(' '.join(t.lemma_ for t in doc if not t.is_space))
    return lemas

print('\nAplicando lematización a train...')
df_train['MESSAGE_LEMMA'] = lematizar_corpus(df_train['MESSAGE_CLEAN'].tolist())

print('Aplicando lematización a test...')
df_test['MESSAGE_LEMMA'] = lematizar_corpus(df_test['MESSAGE_CLEAN'].tolist())

print('\nEjemplos lemma (clean → lemma):')
for i in range(3):
    clean = df_train['MESSAGE_CLEAN'].iloc[i]
    lemma = df_train['MESSAGE_LEMMA'].iloc[i]
    print(f'  CLEAN: {clean[:70]}')
    print(f'  LEMMA: {lemma[:70]}')
    print()

Cargando modelo spaCy...
Modelo cargado.

Aplicando lematización a train...


Lematizando: 100%|██████████| 7196/7196 [00:07<00:00, 912.77it/s] 


Aplicando lematización a test...


Lematizando: 100%|██████████| 1800/1800 [00:01<00:00, 957.57it/s]


Ejemplos lemma (clean → lemma):
  CLEAN: rica económicamente pero muy pobre en objetividad
  LEMMA: rico económicamente pero mucho pobre en objetividad

  CLEAN: en algo tiene razón mafias hay en todo hasta en su 4t
  LEMMA: en algo tener razón mafia haber en todo hasta en su 4 t

  CLEAN: de cuándo acá tan preocupados por la ciencia y la investigación
  LEMMA: de cuándo acá tanto preocupado por el ciencia y el investigación



## 8. Exportación por variante dialectal

Se exportan 18 archivos:

- 3 representaciones (`normal`, `stem`, `lemma`);
- 3 variantes dialectales (`mx`, `es`, `cu`);
- 2 splits (`train`, `test`).

Los tres archivos de train de una misma variante contienen las mismas filas y solo cambia `MESSAGE_CLEAN`.


In [9]:
def exportar_por_variante(df_tr, df_te, col_texto, sufijo):
    cols_base = [
        'ID',
        'TOPIC',
        'IS_IRONIC',
        'MESSAGE',
        'VARIANTE',
    ] + FEATURE_COLS

    for split, df in [
        ('train', df_tr),
        ('test', df_te),
    ]:
        df_out = df[cols_base].copy()
        df_out['MESSAGE_CLEAN'] = df[col_texto]

        for cod in ['mx', 'es', 'cu']:
            (
                df_out[
                    df_out['VARIANTE'] == cod
                ]
                .reset_index(drop=True)
                .to_csv(
                    f'../data/{split}_clean{sufijo}_{cod}.csv',
                    index=False,
                )
            )

    tipo = sufijo if sufijo else '(normal)'
    print(
        f'  {tipo:8} -> '
        f'train_clean{sufijo}_mx/es/cu.csv | '
        f'test_clean{sufijo}_mx/es/cu.csv'
    )


print('Exportando datasets por variante dialectal...\n')

exportar_por_variante(
    df_train,
    df_test,
    'MESSAGE_CLEAN',
    '',
)

exportar_por_variante(
    df_train,
    df_test,
    'MESSAGE_STEM',
    '_stem',
)

exportar_por_variante(
    df_train,
    df_test,
    'MESSAGE_LEMMA',
    '_lemma',
)

print('\n18 archivos generados en ../data/')


# Auditoría interna del TRAIN.
df_auditoria_train.to_csv(
    '../data/auditoria_duplicados_conflictos_train_ml.csv',
    index=False,
)

if not df_conflictos_train.empty:
    df_conflictos_train.to_csv(
        '../data/conflictos_etiqueta_train_ml.csv',
        index=False,
    )

print('Auditoría interna del train guardada en ../data/')


Exportando datasets por variante dialectal...

  (normal) -> train_clean_mx/es/cu.csv | test_clean_mx/es/cu.csv
  _stem    -> train_clean_stem_mx/es/cu.csv | test_clean_stem_mx/es/cu.csv
  _lemma   -> train_clean_lemma_mx/es/cu.csv | test_clean_lemma_mx/es/cu.csv

18 archivos generados en ../data/
Auditoría interna del train guardada en ../data/


## 9. Resumen

Los notebooks de modelos reciben directamente:

- `train_clean_*` / `test_clean_*`;
- `train_clean_stem_*` / `test_clean_stem_*`;
- `train_clean_lemma_*` / `test_clean_lemma_*`.

La separación metodológica queda así:

1. preprocessing → transforma los textos;
2. notebooks ML → construyen `GROUP_ID`;
3. Nested CV 5×3 → `StratifiedGroupKFold`;
4. test oficial → evaluación final.
